Requirements: selenium

In [1]:
# imports
import selenium, time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

Note: Every kayak.com search URL is built like this:

- /flights
- /ORIGIN-DESTINATION
- /YYYY-MM-DD

The part followed after the date will be generated automatically: `?ucs=13qp7li&sort=bestflight_a`

In [2]:
# Create a new driver and scrape an example KAYAK search
driver = webdriver.Chrome()
options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--window-size=1920,1080')  # headless has no window size by default
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36')
driver.get("https://www.kayak.com/flights/BSL-SAW/2026-05-13?ucs=13qp7li&sort=bestflight_a")
time.sleep(10)

### Structure of KAYAK and its flights

Every flight listened on the search is inside a `<div>` with the following class name: `Fxw9-result-item-container`.

While the `Fxw9` part is dynamically changed, `-result-item-container` is always the same.

Ideally, we want to go for this div's via Regex or Xpath.

In [3]:
flights = driver.find_elements('xpath', "//div[contains(@class, '-result-item-container')]")
len(flights)

# I just noticed that Ad's are also inside the same class. Tho we can filter them out because they have a div containing the text "Ad".

flights_cleaned = []

for f in flights:
    is_ad = f.find_elements('xpath', './/div[text()="Ad"]')
    if not is_ad:
        flights_cleaned.append(f)

print(len(flights_cleaned))

20


### Define flight informations we need

Ideally we want to keep the following informations about a flight:

- Departure/Arrival time
- Stopover?
- Origin/Destination
- Flight time
- Baggage included?

For that we have to analyse the div with all the flights we obtained.

After doing so, here are the takeaways:

- **Price**: Is inside a `div` with class name: `e2GB-price-text`
- **Operator**: Is inside a `div` with attribute `dir` named `ltr`
- **Stopover**: Is inside a `div` with class name: `vmXl vmXl-mod-variant-default`
- **Flight time**: Is inside a `div` with class name: `vmXl vmXl-mod-variant-large`
- **Origin/Destination**: We don't need to scrape this information as were providing it ourselfes as parameter for flight search.
- **Baggage**: Is inside a `div` with attribute @aria-label with 'carry-on' as value.

In [4]:
# We have all flights already in our `flights` variable. We will start searching from there for the prices first.
prices = []

for flight in flights:
    try:
        price_element = flight.find_element('xpath', ".//div[contains(@class, '-price-text')]")
        prices.append(price_element.text)
    except:
        continue 

We could extract all the prices by going through our div we found earlier. We are using a try catch to skip anything else which does not contain `-price-text` in its class name to prevent the script from crashing. We can now do the same for the other informations.

In [ ]:
stopovers = []
flight_times = []
operators = []
bags = []

for flight in flights:
    try:
        price_element = flight.find_element('xpath', ".//div[contains(@class, '-price-text')]")
        prices.append(price_element.text)
        stopover = flight.find_element('xpath', ".//div[contains(@class, '-mod-variant-default')]/span")
        stopovers.append(stopover.text)
        flight_time = flight.find_element('xpath', ".//div[contains(@class, '-mod-variant-large')]")
        flight_times.append(flight_time.text)
        operator = flight.find_element('xpath', "//div[contains(@dir, 'ltr')]")
        operators.append(operator.text)
        bag = flight.find_element('xpath', "//div[contains(@aria-label, 'carry-on')]")
        bags.append(bag.text)
    except:
        continue

### Is our information correct?

We gathered most of the info besides the baggages now. We have to check if all lists have the same length. This is important, otherwise we can't zip them together and the data would not match together and be considered as "false information".

In [23]:
print(len(flights))
print(len(flight_times))
print(len(stopovers))
print(len(operators))
print(len(prices))
print(len(bags))
print(operators[0])

24
18
18
18
18
18
easyJet, Pegasus Airlines


Should be correct, as we have on every information we need the same amount. We could start building a flights list:

In [28]:
all_flights = zip(operators, flight_times, stopovers, prices, bags)
type(all_flights)

zip

In [26]:
print("Flights from ORI to DES on 2026-05-23:\n")

for operator, flight_time, stopover, price, bag in all_flights:
    print(f"Operator: {operator}\nFlight Time: {flight_time}\nStopover: {stopover}\nPrice: {price}\nBags included: {bag}\n")
    

Flights from ORI to DES on 2026-05-23:

Operator: easyJet, Pegasus Airlines
Flight Time: 2:35 pm – 9:40 pm
Stopover: 1 stop
Price: $323
Bags included: 0

Operator: easyJet, Pegasus Airlines
Flight Time: 2:35 pm – 4:50 pm
+1
Stopover: 1 stop
Price: $295
Bags included: 0

Operator: easyJet, Pegasus Airlines
Flight Time: 2:45 pm – 10:50 pm
Stopover: 1 stop
Price: $406
Bags included: 0

Operator: easyJet, Pegasus Airlines
Flight Time: 11:00 am – 8:30 pm
Stopover: 1 stop
Price: $350
Bags included: 0

Operator: easyJet, Pegasus Airlines
Flight Time: 9:05 am – 10:30 pm
Stopover: 2 stops
Price: $406
Bags included: 0

Operator: easyJet, Pegasus Airlines
Flight Time: 9:05 am – 9:55 pm
Stopover: 2 stops
Price: $429
Bags included: 0

Operator: easyJet, Pegasus Airlines
Flight Time: 9:05 am – 10:30 pm
Stopover: 2 stops
Price: $409
Bags included: 0

Operator: easyJet, Pegasus Airlines
Flight Time: 9:05 am – 10:10 pm
Stopover: 2 stops
Price: $439
Bags included: 0

Operator: easyJet, Pegasus Airlines
